URBANEV DATASET

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import os

In [102]:
URBAN_DIR   = "UrbanEV_ SZ_districts" 
OUTPUT_PATH = "urbanev_cleaned_processed.csv"

In [3]:
os.makedirs("outputs", exist_ok=True)

In [4]:
def path(filename):
    return os.path.join(URBAN_DIR, filename)

In [6]:
import glob

glob.glob("**/time.csv", recursive=True)

['urban_ev_data\\UrbanEV_ SZ_districts\\time.csv']

In [47]:
df_time = pd.read_csv(
    "urban_ev_data\\UrbanEV_ SZ_districts\\time.csv",
    encoding="utf-8-sig"
)

In [48]:
df_time["timestamp_id"] = range(1, len(df_time) + 1)

In [49]:
df_time["datetime"] = pd.to_datetime(
    df_time[["year", "month", "day", "hour", "minute", "second"]]
)

In [50]:
df_time = df_time[["timestamp_id", "datetime"]].copy()
 
print(f"  Date range: {df_time['datetime'].min()} → {df_time['datetime'].max()}")

  Date range: 2022-06-19 00:00:00 → 2022-07-18 23:55:00


In [51]:
df_time.info()

<class 'pandas.DataFrame'>
RangeIndex: 8640 entries, 0 to 8639
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   timestamp_id  8640 non-null   int64         
 1   datetime      8640 non-null   datetime64[us]
dtypes: datetime64[us](1), int64(1)
memory usage: 135.1 KB


In [12]:
glob.glob("**/information.csv", recursive=True)

['urban_ev_data\\UrbanEV_ SZ_districts\\information.csv']

In [16]:
df_info = pd.read_csv(
    "urban_ev_data\\UrbanEV_ SZ_districts\\information.csv"
   )

In [17]:
df_info = df_info.rename(columns={
    "num"  : "district_num",
    "grid" : "district_id",
    "la"   : "latitude",
    "lon"  : "longitude",
    "count": "total_chargers"
})

In [18]:
print(f"  Districts with dynamic pricing already : {df_info['dynamic_pricing'].sum()}")
print(f"  CBD districts                          : {df_info['CBD'].sum()}")
print(f"  Total chargers across all districts    : {df_info['total_chargers'].sum():,}")

  Districts with dynamic pricing already : 57
  CBD districts                          : 62
  Total chargers across all districts    : 18,061


In [30]:
print(glob.glob("**/occupancy.csv", recursive=True))
print(glob.glob("**/volume.csv", recursive=True))
print(glob.glob("**/price.csv", recursive=True))
print(glob.glob("**/duration.csv", recursive=True))

['urban_ev_data\\UrbanEV_ SZ_districts\\occupancy.csv']
['urban_ev_data\\UrbanEV_ SZ_districts\\volume.csv']
['urban_ev_data\\UrbanEV_ SZ_districts\\price.csv']
['urban_ev_data\\UrbanEV_ SZ_districts\\duration.csv']


In [28]:
df_occ = pd.read_csv(
    r"urban_ev_data\UrbanEV_ SZ_districts\occupancy.csv"
)

In [31]:
df_vol = pd.read_csv(r"urban_ev_data\\UrbanEV_ SZ_districts\\volume.csv")

In [32]:
df_prc  = pd.read_csv(r"urban_ev_data\\UrbanEV_ SZ_districts\\price.csv")

In [33]:
df_dur  = pd.read_csv(r"urban_ev_data\\UrbanEV_ SZ_districts\\duration.csv")

In [37]:
def melt_wide(df_wide, value_name):
    district_cols = [c for c in df_wide.columns if c != "timestamp"]
    df_long = df_wide.melt(
        id_vars=["timestamp"],
        value_vars=district_cols,
        var_name="district_id",
        value_name=value_name
    )

    df_long = df_long.rename(columns={"timestamp": "timestamp_id"})
    df_long["district_id"] = df_long["district_id"].astype(int)
     
    return df_long

     
df_occ_long = melt_wide(df_occ,  "occupancy")
df_vol_long = melt_wide(df_vol,  "volume_kwh")
df_prc_long = melt_wide(df_prc,  "price_cny_per_kwh")
df_dur_long = melt_wide(df_dur,  "avg_duration_hrs")

print(f"  Melted occupancy shape : {df_occ_long.shape}")
print(f"  Expected rows          : 8640 timestamps × 247 districts = {8640*247:,}")


  Melted occupancy shape : (2134080, 3)
  Expected rows          : 8640 timestamps × 247 districts = 2,134,080


MERGING DATA

In [38]:
df = df_occ_long.copy()

In [52]:
df = df.merge(df_vol_long,  on=["timestamp_id", "district_id"], how="left")
df = df.merge(df_prc_long,  on=["timestamp_id", "district_id"], how="left")
df = df.merge(df_dur_long,  on=["timestamp_id", "district_id"], how="left")

df = df.merge(df_time, on="timestamp_id", how="left")

In [53]:
print(df_time.columns)

Index(['timestamp_id', 'datetime'], dtype='str')


In [54]:
print(df_time.head())

   timestamp_id            datetime
0             1 2022-06-19 00:00:00
1             2 2022-06-19 00:05:00
2             3 2022-06-19 00:10:00
3             4 2022-06-19 00:15:00
4             5 2022-06-19 00:20:00


In [55]:
print(df_time.shape)

(8640, 2)


In [56]:
df = df.merge(
    df_info,
    left_on="district_id",
    right_on="district_id",   
    how="left"
)

In [57]:
print(f"  Final merged shape : {df.shape}")
print(f"  Columns            : {df.columns.tolist()}")

  Final merged shape : (2134080, 19)
  Columns            : ['timestamp_id', 'district_id', 'occupancy', 'volume_kwh_x', 'price_cny_per_kwh_x', 'avg_duration_hrs_x', 'volume_kwh_y', 'price_cny_per_kwh_y', 'avg_duration_hrs_y', 'datetime', 'district_num', 'total_chargers', 'fast_count', 'slow_count', 'area', 'longitude', 'latitude', 'CBD', 'dynamic_pricing']


MISSING VALUES

In [ ]:
 df.isnull().sum()
 

timestamp_id           0
district_id            0
occupancy              0
volume_kwh_x           0
price_cny_per_kwh_x    0
avg_duration_hrs_x     0
volume_kwh_y           0
price_cny_per_kwh_y    0
avg_duration_hrs_y     0
datetime               0
district_num           0
total_chargers         0
fast_count             0
slow_count             0
area                   0
longitude              0
latitude               0
CBD                    0
dynamic_pricing        0
dtype: int64

In [61]:

duplicates = df.duplicated().sum()

print("Duplicate Rows:", duplicates)

Duplicate Rows: 0


FEATURE ENGINEERING

In [66]:
df["hour"]  = df["datetime"].dt.hour
df["day_of_week"]  = df["datetime"].dt.dayofweek   # Mon=0, Sun=6
df["is_weekend"]   = (df["day_of_week"] >= 5).astype(int)
df["month"]        = df["datetime"].dt.month
df["day_of_month"] = df["datetime"].dt.day
df["minute"]       = df["datetime"].dt.minute

In [69]:
df["hour_float"]   = df["hour"] + df["minute"] / 60

9:30 WILL BE SHOWN AS 9.5 DUE TO ABOVE CODE

In [70]:
def peak_period(hour):

    if 7 <= hour <= 10:
        return "Morning Peak"

    elif 17 <= hour <= 21:
        return "Evening Peak"

    else:
        return "Off-Peak"

df["peak_period"] = (
    df["hour"].apply(peak_period)
)

In [71]:
df["time_slot"] = pd.cut(
    df["hour"],
    bins=[0, 6, 12, 18, 24],
    labels=["Night(0-6)", "Morning(6-12)", "Afternoon(12-18)", "Evening(18-24)"],
    right=False,
    include_lowest=True
)
 
print("  Extracted: hour, day_of_week, is_weekend, month, is_peak, time_slot")

  Extracted: hour, day_of_week, is_weekend, month, is_peak, time_slot


In [73]:
df["utilization_rate"] = np.where(
    df["total_chargers"] > 0,(df["occupancy"] / df["total_chargers"]).clip(0, 1),0 
    )

In [74]:
print(f"  Avg utilization rate across all districts/times : {df['utilization_rate'].mean():.2%}")
print(f"  Max utilization rate                           : {df['utilization_rate'].max():.2%}")
 

  Avg utilization rate across all districts/times : 28.02%
  Max utilization rate                           : 100.00%


CALCULATE CONGESTION AND UNDERUSE

In [75]:
CONGESTION_THRESHOLD = 0.80  
UNDERUSE_THRESHOLD   = 0.30

In [76]:
df["congestion_flag"] = (df["utilization_rate"] > CONGESTION_THRESHOLD).astype(int)
df["underuse_flag"]   = (df["utilization_rate"] < UNDERUSE_THRESHOLD).astype(int)

In [77]:
pct_congested = df["congestion_flag"].mean() * 100
pct_underused = df["underuse_flag"].mean() * 100
print(f"  % of timesteps with congestion (util > 80%) : {pct_congested:.1f}%")
print(f"  % of timesteps with underuse   (util < 30%) : {pct_underused:.1f}%")

  % of timesteps with congestion (util > 80%) : 1.0%
  % of timesteps with underuse   (util < 30%) : 60.9%


THIS CODE TELLS US HOW MUCH PERCENT OF TIME THE CHARGER IS OVERUSED AND UNDERUSED

In [79]:
print(df.columns)

Index(['timestamp_id', 'district_id', 'occupancy', 'volume_kwh_x',
       'price_cny_per_kwh_x', 'avg_duration_hrs_x', 'volume_kwh_y',
       'price_cny_per_kwh_y', 'avg_duration_hrs_y', 'datetime', 'district_num',
       'total_chargers', 'fast_count', 'slow_count', 'area', 'longitude',
       'latitude', 'CBD', 'dynamic_pricing', 'hour_of_day', 'day_of_week',
       'is_weekend', 'month', 'day_of_month', 'minute', 'hour_float', 'hour',
       'peak_period', 'time_slot', 'utilization_rate', 'congestion_flag',
       'underuse_flag'],
      dtype='str')


In [86]:
df = df.rename(columns={
    "volume_kwh_x": "volume_kwh",
    "price_cny_per_kwh_x": "price_cny_per_kwh",
    "avg_duration_hrs_x": "avg_duration_hrs"
})

In [87]:
duplicate_cols = [
    "volume_kwh_y",
    "price_cny_per_kwh_y",
    "avg_duration_hrs_y"
]


In [88]:
duplicate_cols = [
    col for col in duplicate_cols
    if col in df.columns
]

df = df.drop(columns=duplicate_cols)

In [89]:
print(df.columns)

Index(['timestamp_id', 'district_id', 'occupancy', 'volume_kwh',
       'price_cny_per_kwh', 'avg_duration_hrs', 'datetime', 'district_num',
       'total_chargers', 'fast_count', 'slow_count', 'area', 'longitude',
       'latitude', 'CBD', 'dynamic_pricing', 'hour_of_day', 'day_of_week',
       'is_weekend', 'month', 'day_of_month', 'minute', 'hour_float', 'hour',
       'peak_period', 'time_slot', 'utilization_rate', 'congestion_flag',
       'underuse_flag'],
      dtype='str')


REVENUE

In [90]:
df["revenue_cny"] = df["volume_kwh"] * df["price_cny_per_kwh"]

In [91]:
df["revenue_per_charger_cny"] = np.where(
    df["total_chargers"] > 0,
    df["revenue_cny"] / df["total_chargers"],
    0
)

PRICE DURING BUSY CHARGING TIME MAY INCREASE AT SOME PLACES ,OR MAY REMAIN UNCHANGED 
THIS IS IDENTIFIED BY BELOW CODE

In [93]:
df["pricing_regime"] = np.where(
    df["dynamic_pricing"] == 1, "dynamic", "static"
)

print(f"  Dynamic pricing districts : {df[df['dynamic_pricing']==1]['district_id'].nunique()}")
print(f"  Static  pricing districts : {df[df['dynamic_pricing']==0]['district_id'].nunique()}")

  Dynamic pricing districts : 57
  Static  pricing districts : 190


FAST CHARGER AND SLOW CHARGER

In [97]:
df["fast_charger_dominant"] = (df["fast_count"] > df["slow_count"]).astype(int)



TO LEARN FROM PAST DATA 
LAG1 5 MIN AGO
LAG2 1 HR AGO
LAG3 24 HRS AGO

In [98]:
df = df.sort_values(["district_id", "datetime"]).reset_index(drop=True)
 

df["lag_1"]   = df.groupby("district_id")["utilization_rate"].shift(1)
df["lag_12"]  = df.groupby("district_id")["utilization_rate"].shift(12)
df["lag_288"] = df.groupby("district_id")["utilization_rate"].shift(288)
 

 


In [99]:
df["rolling_3"] = (
    df.groupby("district_id")["utilization_rate"]
    .transform(lambda x: x.rolling(3, min_periods=1).mean())
)

TO DEAL WITH NAN VALUES OF FEW STARTING ROWS

In [100]:
for lag_col in ["lag_1", "lag_12", "lag_288"]:
    df[lag_col] = df.groupby("district_id")[lag_col].transform(
        lambda x: x.fillna(x.mean()))

In [103]:
df.to_csv(OUTPUT_PATH, index=False)